# 3D U-Net + FADC-Bottleneck + Deep Supervision — MAMA-MIA Breast MRI Segmentation

**Model:** UNet3DFADC (fadc_placement='bottleneck', deep_supervision=True) — FADC at bottleneck only + nnU-Net style auxiliary heads at dec2/dec3/dec4.

**Why this experiment:** FADC-Bottleneck without DS landed at Val Dice 0.6851 (+0.0116 over baseline). DS is a standard nnU-Net training trick that typically adds +0.005-0.02 Dice by forcing intermediate decoder features to already form usable predictions. **Note:** DS is a training-trick gain, not architectural — to compare fairly against the no-DS baseline you would also need a `baseline + DS` run.

**DS configuration (nnU-Net default):**
- Main head at `dec1` (128×128×64, weight 1.0)
- Aux head at `dec2` (64×64×32, weight 0.5)
- Aux head at `dec3` (32×32×16, weight 0.25)
- Aux head at `dec4` (16×16×8,  weight 0.125)
- Weights normalized to sum to 1 → [0.5333, 0.2667, 0.1333, 0.0667]
- GT downsampled with `F.interpolate(mode='nearest')`
- Aux heads are active only in `model.train()`; `model.eval()` returns the main head only so MONAI sliding-window inference works unchanged.

**Dataset:** MAMA-MIA (1200 train / 306 val), 2-channel input (pre + post contrast)
**Baseline to beat:** 3D U-Net 2ch Val Dice = 0.6735 (100 epochs)
**Reference (FADC-Bottleneck, no DS):** Val Dice = 0.6851 (100 epochs) — locked 2026-05-27
**Patch size:** 128×128×64
**GPU:** Kaggle T4 x2
**Critical:** 5-epoch LR warmup required for FADC attention modules.

In [ ]:
# ─────────────────────────────────────────────
# CONFIGURATION — edit these before running
# ─────────────────────────────────────────────
# FADC-Bottleneck + Deep Supervision on 2-channel input.

DATA_ROOT    = "/kaggle/input/datasets/bharathvemurik/mama-mia-preprocessed-cache-2ch"
OUTPUT_DIR   = "/kaggle/working/outputs/fadc_bottleneck_ds_2ch_100ep"
CODE_DIR     = "/kaggle/working/FADC-3D"

EPOCHS       = 100
BATCH_SIZE   = 2
NUM_WORKERS  = 4
PATCH_SIZE   = [128, 128, 64]   # FADC only at bottleneck (8x8x4)
WARMUP       = 5                # LR warmup epochs — REQUIRED for FADC attention modules

# Resume from a previous FADC-Bottleneck-DS checkpoint (leave "" for a fresh run)
RESUME_FROM  = ""

# 2-channel preprocessed cache (pre + post contrast).
PREPROCESSED_CACHE_DIR = "/kaggle/input/datasets/bharathvemurik/mama-mia-preprocessed-cache-2ch"

In [ ]:
# ─────────────────────────────────────────────
# 1. INSTALL DEPENDENCIES
# ─────────────────────────────────────────────
import subprocess, sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "monai",
    "--upgrade-strategy", "only-if-needed", "-q"
], check=True)

import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
print("Dependencies ready.")

In [ ]:
# ─────────────────────────────────────────────
# 2. CLONE / UPDATE CODE FROM GITHUB
# ─────────────────────────────────────────────
import os

if os.path.exists(CODE_DIR):
    print("Repo already exists — pulling latest...")
    os.system(f"git -C {CODE_DIR} pull")
else:
    os.system(f"git clone https://github.com/Vemuri-BK/FADC-3D.git {CODE_DIR}")
    print("Repo cloned.")

sys.path.insert(0, CODE_DIR)
print(f"Code path: {CODE_DIR}")

In [ ]:
# ─────────────────────────────────────────────
# 3. VERIFY GPU
# ─────────────────────────────────────────────
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU             : {gpu.name}")
    print(f"VRAM            : {gpu.total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU — enable GPU T4 x2 in Settings → Accelerator")

In [ ]:
# ─────────────────────────────────────────────
# 4. VERIFY 2-CHANNEL .npz CACHE
# ─────────────────────────────────────────────
import os, numpy as np
from pathlib import Path

cache_path = Path(PREPROCESSED_CACHE_DIR)
print(f"Cache path        : {cache_path}")
print(f"Cache path exists : {cache_path.exists()}")
assert cache_path.exists(), f"Cache not found — check that dataset is attached: {cache_path}"

train_npz = sorted((cache_path / "train").glob("*.npz")) if (cache_path / "train").exists() else []
val_npz   = sorted((cache_path / "val").glob("*.npz"))   if (cache_path / "val").exists()   else []
print(f"Train .npz files  : {len(train_npz)}")
print(f"Val   .npz files  : {len(val_npz)}")
assert len(train_npz) > 0 and len(val_npz) > 0, "Cache is empty — train/ and val/ have no .npz files."

collections = {}
for p in train_npz:
    col = p.stem.split("_")[0].upper()
    collections[col] = collections.get(col, 0) + 1
print("\nTrain per-collection:")
for col, count in sorted(collections.items()):
    print(f"  {col}: {count} cases")

In [ ]:
# -----------------------------------------------
# 4b. SANITY-CHECK 2-CHANNEL CACHE
#     Verify .npz files have shape (2, H, W, D) before launching training.
# -----------------------------------------------
import os, numpy as np
from pathlib import Path

assert PREPROCESSED_CACHE_DIR, "PREPROCESSED_CACHE_DIR is empty — set it in the config cell."
cache_path = Path(PREPROCESSED_CACHE_DIR)
assert cache_path.exists(), f"Cache path not found: {cache_path}"

train_npzs = sorted((cache_path / "train").glob("*.npz"))
val_npzs   = sorted((cache_path / "val").glob("*.npz"))
print(f"Train .npz files : {len(train_npzs)}")
print(f"Val   .npz files : {len(val_npzs)}")
assert len(train_npzs) > 0 and len(val_npzs) > 0, "No .npz files found in train/ or val/ subdirs."

sample_paths = [train_npzs[0], train_npzs[len(train_npzs)//2], train_npzs[-1], val_npzs[0]]
for p in sample_paths:
    d = np.load(p)
    img_shape = d["image"].shape
    lbl_shape = d["label"].shape
    print(f"  {p.name:35s}  image={img_shape}  label={lbl_shape}  img_dtype={d['image'].dtype}")
    assert img_shape[0] == 2, (
        f"FATAL: {p.name} has {img_shape[0]} channels, expected 2. "
        f"You mounted a 1-channel cache — re-mount the -2ch dataset before training."
    )
    assert lbl_shape[0] == 1, f"Label channel mismatch in {p.name}: {lbl_shape}"

print("\nAll sampled cases are 2-channel — cache looks good. Safe to launch training.")

In [ ]:
# -----------------------------------------------
# 4c. SANITY-CHECK DS MODEL FORWARD
#     Confirm the cloned repo has the deep_supervision wiring in place
#     before launching the 8h run.
# -----------------------------------------------
import torch
import sys
sys.path.insert(0, CODE_DIR)
from models.unet_3d_fadc import UNet3DFADC

m = UNet3DFADC(in_channels=2, out_channels=2, base_filters=32,
               fadc_placement='bottleneck', deep_supervision=True)

# Train mode: must return 4-tuple at decreasing scales
m.train()
x = torch.randn(1, 2, 128, 128, 64)
with torch.no_grad():
    y = m(x)
assert isinstance(y, tuple) and len(y) == 4, f"train mode must return 4-tuple, got {type(y)}"
expected = [(1,2,128,128,64), (1,2,64,64,32), (1,2,32,32,16), (1,2,16,16,8)]
for i, (t, exp) in enumerate(zip(y, expected)):
    assert tuple(t.shape) == exp, f"head {i}: got {t.shape}, expected {exp}"
print("DS train mode shapes:", [tuple(t.shape) for t in y], "OK")

# Eval mode: must return single tensor (for sliding-window inference)
m.eval()
with torch.no_grad():
    y_eval = m(x)
assert isinstance(y_eval, torch.Tensor) and y_eval.shape == (1, 2, 128, 128, 64), \
    f"eval mode must return single tensor, got {type(y_eval)}"
print("DS eval mode shape :", y_eval.shape, "OK")

n = sum(p.numel() for p in m.parameters())
print(f"Total params (Bottleneck + DS): {n:,}")
del m, y, y_eval, x

In [ ]:
# ─────────────────────────────────────────────
# 5. RUN TRAINING — FADC-Bottleneck + Deep Supervision
# ─────────────────────────────────────────────
import os, subprocess, sys
os.makedirs(OUTPUT_DIR, exist_ok=True)

train_script = os.path.join(CODE_DIR, "training", "train_centralized.py")

cmd = [
    sys.executable, "-u", train_script,
    "--model",            "unet3d_fadc_bottleneck",
    "--deep_supervision",
    "--data_root",        DATA_ROOT,
    "--output_dir",       OUTPUT_DIR,
    "--epochs",           str(EPOCHS),
    "--batch_size",       str(BATCH_SIZE),
    "--num_workers",      str(NUM_WORKERS),
    "--patch_size",       str(PATCH_SIZE[0]), str(PATCH_SIZE[1]), str(PATCH_SIZE[2]),
    "--warmup_epochs",    str(WARMUP),
]

if RESUME_FROM:
    cmd += ["--resume", RESUME_FROM]

if PREPROCESSED_CACHE_DIR:
    cmd += ["--preprocessed_cache_dir", PREPROCESSED_CACHE_DIR]

print("Command:", " ".join(cmd))
print("=" * 60)

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
while True:
    chunk = process.stdout.read(512)
    if not chunk:
        break
    sys.stdout.write(chunk.decode("utf-8", errors="replace"))
    sys.stdout.flush()
process.wait()
print(f"\nExit code: {process.returncode}")

In [ ]:
# ─────────────────────────────────────────────
# 6. PLOT TRAINING CURVES
# ─────────────────────────────────────────────
import json
import matplotlib.pyplot as plt

BASELINE_DICE    = 0.6735   # 3D U-Net 2ch, 100 epochs (locked 2026-05-24)
BOTTLENECK_DICE  = 0.6851   # FADC-Bottleneck 2ch, no DS, 100 epochs (locked 2026-05-27)
PAPER_DICE       = 0.762    # MAMA-MIA paper nnU-Net reference

log_path = os.path.join(OUTPUT_DIR, "train_log.json")

if not os.path.exists(log_path):
    print("No training log found yet.")
else:
    with open(log_path) as f:
        log = json.load(f)

    epochs     = [e["epoch"]    for e in log]
    losses     = [e["loss"]     for e in log]
    val_epochs = [e["epoch"]    for e in log if "val_dice" in e]
    val_dices  = [e["val_dice"] for e in log if "val_dice" in e]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(epochs, losses, color="steelblue", linewidth=1.5)
    ax1.set_title("Training Loss (weighted DS total)", fontsize=13)
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.grid(True, alpha=0.3)

    ax2.plot(val_epochs, val_dices, color="purple", linewidth=1.5, marker="o", markersize=4)
    ax2.axhline(y=BASELINE_DICE,   color="green",  linestyle="--", linewidth=1.5, label=f"3D U-Net 2ch baseline ({BASELINE_DICE:.4f})")
    ax2.axhline(y=BOTTLENECK_DICE, color="orange", linestyle="--", linewidth=1.5, label=f"FADC-Bottleneck no-DS ({BOTTLENECK_DICE:.4f})")
    ax2.axhline(y=PAPER_DICE,      color="red",    linestyle="--", linewidth=1,   label=f"MAMA-MIA paper ({PAPER_DICE:.3f})")
    ax2.set_title("Validation Dice (main head)", fontsize=13)
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Dice Score")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    if val_dices:
        best = max(val_dices)
        ax2.set_title(f"Validation Dice  (best: {best:.4f})", fontsize=13)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150)
    plt.show()
    print(f"Epochs completed     : {len(log)}")
    if val_dices:
        print(f"Best Val Dice        : {max(val_dices):.4f}")
        print(f"3D U-Net baseline    : {BASELINE_DICE:.4f}")
        print(f"FADC-Bottleneck no-DS: {BOTTLENECK_DICE:.4f}")
        print(f"vs baseline          : {max(val_dices) - BASELINE_DICE:+.4f}")
        print(f"vs Bottleneck no-DS  : {max(val_dices) - BOTTLENECK_DICE:+.4f}")

In [ ]:
# ─────────────────────────────────────────────
# 7. SEGMENTATION VISUALIZATION
#    Run after training completes (best_model.pth must exist).
#    DS aux heads exist in the checkpoint state_dict, so we instantiate
#    the model with deep_supervision=True and let eval() return main only.
# ─────────────────────────────────────────────
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import torch
from matplotlib.patches import Patch

sys.path.insert(0, CODE_DIR)
from models.unet_3d_fadc import UNet3DFADC
from data.mama_mia_dataset import build_centralized_loaders
from monai.inferers import sliding_window_inference
from monai.transforms import AsDiscrete

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

best_ckpt_path = os.path.join(OUTPUT_DIR, "best_model.pth")
ckpt = torch.load(best_ckpt_path, map_location=device)
cfg  = ckpt["config"]

# Read DS flag from saved cfg so loading still works if you later toggle it.
ds_flag = cfg["model"].get("deep_supervision", False)

model = UNet3DFADC(
    in_channels      = cfg["model"]["in_channels"],
    out_channels     = cfg["model"]["out_channels"],
    base_filters     = cfg["model"]["base_filters"],
    fadc_placement   = 'bottleneck',
    deep_supervision = ds_flag,
).to(device)
model.load_state_dict(ckpt["model"])
model.eval()   # eval mode forces main-head-only forward, even with DS heads present

BASELINE_DICE   = 0.6735
BOTTLENECK_DICE = 0.6851
print(f"Loaded FADC-Bottleneck+DS best model — Epoch {ckpt['epoch']+1} | Val Dice: {ckpt['best_dice']:.4f}")
print(f"vs 3D U-Net 2ch baseline ({BASELINE_DICE:.4f})       : {ckpt['best_dice'] - BASELINE_DICE:+.4f}")
print(f"vs FADC-Bottleneck no-DS ({BOTTLENECK_DICE:.4f}): {ckpt['best_dice'] - BOTTLENECK_DICE:+.4f}")

split_csv = os.path.join(DATA_ROOT, "train_test_splits.csv")
_, val_loader = build_centralized_loaders(
    data_root              = DATA_ROOT,
    split_csv              = split_csv if os.path.exists(split_csv) else None,
    cache_rate             = 0.0,
    num_workers            = 0,
    batch_size             = 1,
    preprocessed_cache_dir = PREPROCESSED_CACHE_DIR,
    patch_size             = tuple(cfg["data"]["patch_size"]),
)

post_pred  = AsDiscrete(argmax=True)
PATCH_SIZE_TUPLE = tuple(cfg["data"]["patch_size"])

N_CASES = 5
results = []

with torch.no_grad():
    for batch in val_loader:
        if len(results) >= N_CASES:
            break
        images = batch["image"].to(device)
        labels = batch["label"]
        preds    = sliding_window_inference(images, PATCH_SIZE_TUPLE, 4, model, overlap=0.25)
        pred_cls = post_pred(preds[0]).cpu().numpy()
        pred_fg  = (pred_cls[0] == 1).astype(np.float32)
        label_fg = labels[0, 0].numpy()
        image_np = images[0, 1].cpu().numpy() if images.shape[1] >= 2 else images[0, 0].cpu().numpy()
        tp    = (pred_fg * label_fg).sum()
        denom = pred_fg.sum() + label_fg.sum()
        dice  = float(2 * tp / (denom + 1e-6))
        sums = label_fg.sum(axis=(0, 1))
        z    = int(sums.argmax()) if sums.max() > 0 else image_np.shape[2] // 2
        patient = batch.get("patient_id", ["unknown"])[0] if isinstance(batch, dict) else "?"
        results.append({"image": image_np, "gt": label_fg, "pred": pred_fg,
                        "dice": dice, "z": z, "patient": patient})

print(f"Inference done on {len(results)} validation cases")

fig, axes = plt.subplots(len(results), 4, figsize=(20, 5 * len(results)))
if len(results) == 1:
    axes = axes[np.newaxis, :]

COL_TITLES = ["MRI Input (post-contrast)", "Ground Truth (green)", "Prediction (cyan)", "TP / FP / FN"]

for i, r in enumerate(results):
    img, gt, pred, z = r["image"], r["gt"], r["pred"], r["z"]
    img_norm = (img - img.min()) / (img.max() - img.min() + 1e-8)
    img_s  = img_norm[:, :, z].T
    gt_s   = gt[:, :, z].T
    pred_s = pred[:, :, z].T

    axes[i, 0].imshow(img_s, cmap="gray", origin="lower")
    axes[i, 0].set_ylabel(f"{r['patient']}\nDice={r['dice']:.3f}", fontsize=9)

    axes[i, 1].imshow(img_s, cmap="gray", origin="lower")
    gt_rgba = np.zeros((*gt_s.shape, 4))
    gt_rgba[gt_s > 0] = [0.0, 1.0, 0.0, 0.55]
    axes[i, 1].imshow(gt_rgba, origin="lower")

    axes[i, 2].imshow(img_s, cmap="gray", origin="lower")
    pred_rgba = np.zeros((*pred_s.shape, 4))
    pred_rgba[pred_s > 0] = [0.0, 0.85, 1.0, 0.55]
    axes[i, 2].imshow(pred_rgba, origin="lower")

    axes[i, 3].imshow(img_s, cmap="gray", origin="lower")
    overlay = np.zeros((*gt_s.shape, 4))
    overlay[(gt_s > 0) & (pred_s > 0)] = [1.00, 1.00, 0.00, 0.65]
    overlay[(gt_s == 0) & (pred_s > 0)] = [1.00, 0.20, 0.20, 0.65]
    overlay[(gt_s > 0) & (pred_s == 0)] = [0.20, 1.00, 0.20, 0.65]
    axes[i, 3].imshow(overlay, origin="lower")

    for ax in axes[i]:
        ax.axis("off")

for j, title in enumerate(COL_TITLES):
    axes[0, j].set_title(title, fontsize=11, fontweight="bold", pad=8)

legend_handles = [
    Patch(facecolor="yellow", label="True Positive  (TP)"),
    Patch(facecolor="red",    label="False Positive (FP)"),
    Patch(facecolor="lime",   label="False Negative (FN)"),
]
fig.legend(handles=legend_handles, loc="lower center", ncol=3,
           fontsize=10, bbox_to_anchor=(0.5, -0.01), framealpha=0.95)

plt.suptitle(
    f"FADC-Bottleneck + DS Results — Best Model (Val Dice: {ckpt['best_dice']:.4f} | Epoch {ckpt['epoch']+1})",
    fontsize=13, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "segmentation_results.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Visualization saved to:", OUTPUT_DIR)